In [0]:
"""
02_build_master_transactions

This notebook reads validated Olist datasets from ADLS Gen2, performs business
transformations and enrichments, creates a curated master_transactions table,
and stores the final output as a Delta Lake dataset in the processed layer for
downstream analytics and reporting.
"""

'\n02_build_master_transactions\n\nThis notebook reads validated Olist datasets from ADLS Gen2, performs business\ntransformations and enrichments, creates a curated master_transactions table,\nand stores the final output as a Delta Lake dataset in the processed layer for\ndownstream analytics and reporting.\n'

In [0]:
storage_account_name = "shopscope2026"

# Pull credentials from Databricks secret scope instead of hardcoding them
client_id = dbutils.secrets.get(scope="shopscope-kv", key="client-id")
tenant_id = dbutils.secrets.get(scope="shopscope-kv", key="tenant-id")
client_secret = dbutils.secrets.get(scope="shopscope-kv", key="client-secret")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("ADLS Authentication Configured")

ADLS Authentication Configured


In [0]:
raw_path = "abfss://raw@shopscope2026.dfs.core.windows.net"

orders_df = spark.read.option("header", "true").csv(
    f"{raw_path}/orders/olist_orders_dataset.csv"
)

customers_df = spark.read.option("header", "true").csv(
    f"{raw_path}/customers/olist_customers_dataset.csv"
)

payments_df = spark.read.option("header", "true").csv(
    f"{raw_path}/payments/olist_order_payments_dataset.csv"
)

order_items_df = spark.read.option("header", "true").csv(
    f"{raw_path}/order_items/olist_order_items_dataset.csv"
)

products_df = spark.read.option("header", "true").csv(
    f"{raw_path}/products/olist_products_dataset.csv"
)

reviews_df = spark.read.option("header", "true").csv(
    f"{raw_path}/reviews/olist_order_reviews_dataset.csv"
)

category_names_df = spark.read.option("header", "true").csv(
    f"{raw_path}/category_names/product_category_name_translation.csv"
)

print("Orders:", orders_df.count())
print("Customers:", customers_df.count())
print("Payments:", payments_df.count())
print("Order Items:", order_items_df.count())
print("Products:", products_df.count())
print("Reviews:", reviews_df.count())
print("Category Names:", category_names_df.count())

Orders: 99441
Customers: 99441
Payments: 103886
Order Items: 112650
Products: 32951
Reviews: 104162
Category Names: 71


In [0]:
from pyspark.sql.functions import col, to_timestamp, regexp_replace, when, lit

orders_clean = (
    orders_df
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at")))
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date")))
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date")))
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date")))
)

order_items_clean = (
    order_items_df
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
    .withColumn("order_item_id", col("order_item_id").cast("int"))
)

payments_clean = (
    payments_df
    .withColumn("payment_sequential", col("payment_sequential").cast("int"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
    .withColumn("payment_value", col("payment_value").cast("double"))
)

reviews_clean = reviews_df.withColumn("review_score", col("review_score").cast("int"))

products_clean = products_df
customers_clean = customers_df
category_names_clean = category_names_df

print("Data cleaning complete")

Data cleaning complete


In [0]:
from pyspark.sql.functions import sum as _sum, count as _count

order_aggregates_df = (
    order_items_clean
    .groupBy("order_id")
    .agg(
        _sum("price").alias("total_order_value"),
        _sum("freight_value").alias("total_freight"),
        _count("*").alias("item_count")
    )
)

In [0]:
from pyspark.sql.functions import first

payment_aggregates_df = (
    payments_clean
    .groupBy("order_id")
    .agg(
        _sum("payment_value").alias("total_payment_value"),
        first("payment_type", ignorenulls=True).alias("payment_type"),
        _sum("payment_installments").alias("payment_installments")
    )
)

In [0]:
reviews_enriched_df = (
    reviews_clean
    .withColumn(
        "sentiment_category",
        when(col("review_score").between(4, 5), lit("Positive"))
        .when(col("review_score") == 3, lit("Neutral"))
        .when(col("review_score").between(1, 2), lit("Negative"))
        .otherwise(lit("Unknown"))
    )
    .select("order_id", "review_score", "sentiment_category")
)

In [0]:
products_enriched_df = (
    products_clean
    .join(category_names_clean, on="product_category_name", how="left")
    .withColumnRenamed("product_category_name_english", "category_en")
)

item_product_df = (
    order_items_clean
    .join(products_enriched_df, on="product_id", how="left")
)

In [0]:
from pyspark.sql.functions import first

product_features_by_order_df = (
    item_product_df
    .groupBy("order_id")
    .agg(
        first("product_id", ignorenulls=True).alias("product_id"),
        first("product_category_name", ignorenulls=True).alias("product_category_name"),
        first("category_en", ignorenulls=True).alias("category_en")
    )
)

master_transactions_df = (
    orders_clean
    .join(customers_clean, on="customer_id", how="left")
    .join(order_aggregates_df, on="order_id", how="left")
    .join(payment_aggregates_df, on="order_id", how="left")
    .join(reviews_enriched_df, on="order_id", how="left")
    .join(product_features_by_order_df, on="order_id", how="left")
)

In [0]:
from pyspark.sql.functions import year, month, dayofmonth, datediff

master_transactions_df = (
    master_transactions_df
    .withColumn("purchase_year", year(col("order_purchase_timestamp")))
    .withColumn("purchase_month", month(col("order_purchase_timestamp")))
    .withColumn("purchase_day", dayofmonth(col("order_purchase_timestamp")))
    .withColumn(
        "delivery_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    )
)

In [0]:
processed_output_path = "abfss://processed@shopscope2026.dfs.core.windows.net/master_transactions"

master_transactions_df.write.format("delta").mode("overwrite").save(processed_output_path)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7746402853088935>, line 3
      1 processed_output_path = "abfss://processed@shopscope2026.dfs.core.windows.net/master_transactions"
----> 3 master_transactions_df.write.format("delta").mode("overwrite").save(processed_output_path)

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/spark/python/pyspark/sql/connect/client/core.py:1589, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1587     req.user_context.user_id = self._user_id
   1588 

In [0]:
saved_df = spark.read.format("delta").load(processed_output_path)

print("Rows:", saved_df.count())
print("Columns:", len(saved_df.columns))
display(saved_df.limit(10))